# CAP Sleep Database: preprocessing

EEG (C4-A1 / C3-A2) and chin EMG are band-pass filtered, resampled to 125 Hz and cut into 5-s windows aligned with the sleep-stage annotations.
Labels come from the ECG: HR (5-s window) and RMSSD (60-s context). Windows without a valid ECG label or with a flat channel are removed; clipped windows are flagged and kept.
Output: `sleep_project/cap_processed/<record>.npz` (103 records; nfle27 excluded by design, nfle12 skipped because its ECG has no usable R peaks).

In [ ]:
# Settings
ROOT = "/content/drive/MyDrive/sleep_project"
FORCE_REPROCESS = False
N_DOWNLOAD_THREADS = 6

In [ ]:
# Google Drive, packages and folders
from google.colab import drive
drive.mount("/content/drive")
!pip install -q neurokit2

import os, sys, re, json, glob, time, shutil, hashlib, threading, importlib
import numpy as np, pandas as pd, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

S3 = "https://physionet-open.s3.amazonaws.com/capslpdb/1.0.0/"
EDF = Path(ROOT) / "cap_edf"
ANN = Path(ROOT) / "cap_annotations"
OUT = Path(ROOT) / "cap_processed"
for p in (EDF, ANN, OUT): p.mkdir(parents=True, exist_ok=True)
print("Drive ready:", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 17.0 MB/s eta 0:00:00
Drive ready: /content/drive/MyDrive/sleep_project


In [ ]:
%%writefile /content/drive/MyDrive/sleep_project/cap_preprocess.py
# Preprocessing module: EDF reading, channel selection, filtering, 5-s windows, ECG-derived HR / RMSSD labels
import re, json, time, warnings
import numpy as np, pandas as pd
from pathlib import Path
from fractions import Fraction
from scipy.signal import butter, sosfiltfilt, resample_poly
import neurokit2 as nk

CODE_VERSION = "v3"
FS = 125
WIN = 5
CTX = 60
BANDS = {"eeg": (0.5, 30), "emg": (10, 60), "ecg": (0.5, 30)}
SLEEP = {"S1": "N1", "S2": "N2", "S3": "N3", "S4": "N3", "REM": "REM"}
GROUP_NAMES = {"n": "Healthy", "nfle": "NFLE", "rbd": "RBD", "plm": "PLM", "ins": "Insomnia",
               "narco": "Narcolepsy", "sdb": "SDB", "brux": "Bruxism"}
EXCLUDE = {"nfle27"}

def read_header(path):
    with open(path, "rb") as f:
        b = f.read(256); ns = int(b[252:256].decode().strip()); f.seek(0); b = f.read(256 + 256 * ns)
    a = lambda x: x.decode("latin-1", "ignore").strip()
    h = {"start_time": a(b[176:184]), "hb": int(a(b[184:192])), "rec_dur": float(a(b[244:252])), "ns": ns}
    pos = 256
    def field(w, conv=a):
        nonlocal pos
        v = [conv(b[pos + i * w: pos + (i + 1) * w]) for i in range(ns)]; pos += w * ns; return v
    num = lambda x: float(a(x) or 0)
    h["labels"] = field(16); field(80); h["unit"] = field(8)
    h["pmin"], h["pmax"], h["dmin"], h["dmax"] = field(8, num), field(8, num), field(8, num), field(8, num)
    field(80); h["spr"] = [int(v) for v in field(8, num)]
    h["n_rec"] = (Path(path).stat().st_size - h["hb"]) // (2 * sum(h["spr"]))
    return h

def read_channel(path, h, i):
    mm = np.memmap(path, dtype="<i2", mode="r", offset=h["hb"], shape=(h["n_rec"], sum(h["spr"])))
    s = sum(h["spr"][:i])
    x = np.asarray(mm[:, s:s + h["spr"][i]], dtype=np.float64).ravel()
    g = (h["pmax"][i] - h["pmin"][i]) / (h["dmax"][i] - h["dmin"][i])
    x = (x - h["dmin"][i]) * g + h["pmin"][i]
    if h["unit"][i].strip().lower() == "mv": x = x * 1000.0
    return x, h["spr"][i] / h["rec_dur"]

norm = lambda l: re.sub(r"[^A-Z0-9]", "", l.upper())
def pick(labels, rules):
    N = [norm(l) for l in labels]
    for rule in rules:
        for i, n in enumerate(N):
            if rule(n): return i
    return None
RULES = {
    "eeg": [lambda n: n in ("C4A1", "C3A2", "C4M1", "C3M2"), lambda n: n.startswith(("C4", "C3"))],
    "emg": [lambda n: n in ("EMG1EMG2", "EMGEMG", "CHIN1", "CHIN", "EMG"), lambda n: n.startswith(("CHIN", "EMG"))],
    "ecg": [lambda n: n in ("ECG1ECG2", "ECG", "EKG"), lambda n: "ECG" in n or "EKG" in n],
}

def to_fs(x, fs):
    if abs(fs - FS) < 1e-6: return x
    fr = Fraction(FS / fs).limit_denominator(2000)
    return resample_poly(x, fr.numerator, fr.denominator)

def bandpass(x, lo, hi):
    sos = butter(2, [lo, min(hi, FS / 2 - 1)], btype="band", fs=FS, output="sos")
    return sosfiltfilt(sos, x)

def to_sec(s):
    p = [int(v) for v in re.split(r"[:.]", s.strip()) if v.isdigit()]
    return p[0] * 3600 + p[1] * 60 + p[2] if len(p) >= 3 else None

def parse_annotation(path):
    lines = Path(path).read_text(encoding="latin-1").splitlines()
    hi = next(i for i, l in enumerate(lines) if "Event" in l and "Time" in l)
    cols = [c.strip().lower() for c in lines[hi].split("\t")]
    ci_t = next(i for i, c in enumerate(cols) if "time" in c)
    ci_e = next(i for i, c in enumerate(cols) if "event" in c)
    ci_d = next(i for i, c in enumerate(cols) if "duration" in c)
    rows, prev, day = [], None, 0
    for l in lines[hi + 1:]:
        p = l.split("\t")
        if len(p) <= max(ci_t, ci_e, ci_d): continue
        t = to_sec(p[ci_t])
        if t is None: continue
        if prev is not None and t + day < prev - 12 * 3600: day += 86400
        t += day; prev = t
        try: d = float(p[ci_d])
        except ValueError: d = 30.0
        rows.append((t, p[ci_e].strip().upper(), d))
    return pd.DataFrame(rows, columns=["t", "event", "dur"])

def load_demographics(xlsx):
    d = pd.read_excel(xlsx, sheet_name=0, header=None, names=["record", "sex", "age"])
    d["record"] = (d.record.astype(str).str.strip().str.lower().str.replace(" ", "")
                   .str.replace(r"^sbd", "sdb", regex=True).str.replace(r"^rdb", "rbd", regex=True))
    d["sex"] = d.sex.astype(str).str.strip().str.upper()
    d["age"] = pd.to_numeric(d.age, errors="coerce")
    return d.set_index("record")

def rr_series(ecg):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        clean = nk.ecg_clean(ecg, sampling_rate=FS, method="neurokit")
        _, info = nk.ecg_peaks(clean, sampling_rate=FS, method="neurokit")
    peaks = np.asarray(info["ECG_R_Peaks"], dtype=int)
    t = peaks / FS
    rr = np.diff(t)
    ok = (rr > 0.3) & (rr < 2.0)
    med = pd.Series(rr).rolling(11, center=True, min_periods=3).median().values
    ok &= np.abs(rr - med) < 0.2 * med
    return t[1:], rr, ok

def window_labels(starts, rr_t, rr, ok):
    hr, rmssd = np.full(len(starts), np.nan), np.full(len(starts), np.nan)
    for k, s in enumerate(starts):
        m = (rr_t >= s) & (rr_t < s + WIN) & ok
        if m.sum() >= 3:
            hr[k] = 60.0 / rr[m].mean()
        c0, c1 = s + WIN / 2 - CTX / 2, s + WIN / 2 + CTX / 2
        idx = np.where((rr_t >= c0) & (rr_t < c1))[0]
        if len(idx) >= 20:
            pair = ok[idx[1:]] & ok[idx[:-1]] & (np.diff(idx) == 1)
            if pair.sum() >= 15:
                d = np.diff(rr[idx])[pair]
                rmssd[k] = np.sqrt(np.mean(d ** 2)) * 1000.0
    return hr, rmssd

def process_record(rec, edf_dir, ann_dir, demo, out_dir):
    out = Path(out_dir) / f"{rec}.npz"
    if out.exists():
        try:
            if json.loads(str(np.load(out)["meta"])).get("code_version") == CODE_VERSION:
                return "exists"
        except Exception:
            pass
        out.unlink()
    edf = Path(edf_dir) / f"{rec}.edf"
    h = read_header(edf)
    idx = {k: pick(h["labels"], r) for k, r in RULES.items()}
    if any(v is None for v in idx.values()):
        return f"skip: missing {[k for k, v in idx.items() if v is None]}"

    sig, raw = {}, {}
    for k, i in idx.items():
        x, fs = read_channel(edf, h, i)
        scale = 1000.0 if h["unit"][i].strip().lower() == "mv" else 1.0
        raw[k] = (x, fs, max(abs(h["pmin"][i]), abs(h["pmax"][i])) * scale)
        sig[k] = bandpass(to_fs(x, fs), *BANDS[k])
        sig[k + "_fs_orig"] = fs
    n = min(len(sig["eeg"]), len(sig["emg"]), len(sig["ecg"]))
    dur = n / FS

    ann = parse_annotation(Path(ann_dir) / f"{rec}.txt")
    hyp = ann[ann.event.str.startswith("SLEEP-")].copy()
    hyp["stage"] = hyp.event.str.replace("SLEEP-", "", regex=False).map(SLEEP)
    hyp["off"] = (hyp.t - to_sec(h["start_time"].replace(".", ":"))) % 86400
    hyp = hyp.dropna(subset=["stage"])
    starts, stages = [], []
    for _, e in hyp.iterrows():
        for j in range(int(round(e.dur / WIN))):
            s = e.off + j * WIN
            if s + WIN <= dur: starts.append(s); stages.append(e.stage)
    starts, stages = np.array(starts), np.array(stages)
    if len(starts) == 0: return "skip: no sleep windows inside the recording"

    a_idx = (starts * FS).astype(int)
    seg = lambda x: np.stack([x[a:a + WIN * FS] for a in a_idx])
    eeg_w, emg_w, ecg_w = seg(sig["eeg"]), seg(sig["emg"]), seg(sig["ecg"])

    flat_eeg = eeg_w.std(1) < 0.5
    flat_emg = emg_w.std(1) < 0.05
    flat_ecg = ecg_w.std(1) < 1e-3
    clip = {}
    for k in ("eeg", "emg"):
        x, fs0, lim = raw[k]
        L = int(round(WIN * fs0))
        clip[k] = np.array([np.mean(np.abs(x[int(s * fs0): int(s * fs0) + L]) >= 0.99 * lim) for s in starts])
    extreme = np.abs(eeg_w).max(1) > 500

    rr_t, rr, ok = rr_series(sig["ecg"][:n])
    hr, rmssd = window_labels(starts, rr_t, rr, ok)
    no_label = np.isnan(hr) | np.isnan(rmssd)

    keep = ~(flat_eeg | flat_emg | flat_ecg | no_label)
    reasons = {"flat_eeg": int(flat_eeg.sum()), "flat_emg": int(flat_emg.sum()), "flat_ecg": int(flat_ecg.sum()),
               "no_label": int(no_label.sum()), "eeg_clip>1%": int((clip["eeg"] > 0.01).sum()),
               "emg_clip>1%": int((clip["emg"] > 0.01).sum()), "eeg>500uV": int(extreme.sum())}
    if keep.sum() == 0: return f"skip: no valid windows {reasons}"

    mu_e, sd_e = eeg_w[keep].mean(), eeg_w[keep].std()
    mu_m, sd_m = emg_w[keep].mean(), emg_w[keep].std()
    eeg_z = ((eeg_w[keep] - mu_e) / sd_e).astype(np.float16)
    emg_z = ((emg_w[keep] - mu_m) / sd_m).astype(np.float16)

    g = re.match(r"[a-z]+", rec).group()
    meta = {"code_version": CODE_VERSION, "record": rec, "group": GROUP_NAMES.get(g, g),
            "age": float(demo.age.get(rec, np.nan)), "sex": str(demo.sex.get(rec, "")),
            "eeg_label": h["labels"][idx["eeg"]], "emg_label": h["labels"][idx["emg"]], "ecg_label": h["labels"][idx["ecg"]],
            "fs_orig": {k: sig[k + "_fs_orig"] for k in ("eeg", "emg", "ecg")},
            "windows_total": int(len(starts)), "windows_kept": int(keep.sum()), "reasons": reasons,
            "eeg_mean_std": [float(mu_e), float(sd_e)], "emg_mean_std": [float(mu_m), float(sd_m)]}
    np.savez_compressed(out, eeg=eeg_z, emg=emg_z,
                        hr=hr[keep].astype(np.float32), rmssd=rmssd[keep].astype(np.float32),
                        stage=stages[keep], t=starts[keep].astype(np.float32),
                        eeg_clip=clip["eeg"][keep].astype(np.float16), emg_clip=clip["emg"][keep].astype(np.float16),
                        eeg_extreme=extreme[keep], meta=json.dumps(meta))
    clipped = int(((clip["eeg"] > 0.01) | (clip["emg"] > 0.01))[keep].sum())
    return f"ok: {keep.sum()}/{len(starts)} windows kept ({clipped} of them clipped >1%)"

def run(records, edf_dir, ann_dir, xlsx, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    demo = load_demographics(xlsx)
    log = []
    for i, rec in enumerate(records, 1):
        t0 = time.time()
        if rec in EXCLUDE:
            status = "skip: excluded"
        else:
            try: status = process_record(rec, edf_dir, ann_dir, demo, out_dir)
            except Exception as e: status = f"ERROR: {type(e).__name__}: {e}"
        log.append({"record": rec, "status": status, "sec": round(time.time() - t0, 1)})
        print(f"[{i:3d}/{len(records)}] {rec:8s} {status}  ({time.time()-t0:.0f}s)", flush=True)
    new = pd.DataFrame(log)
    log_file = Path(out_dir) / "preprocess_log.csv"
    if log_file.exists():
        old = pd.read_csv(log_file)
        new_real = new[new.status != "exists"]
        merged = pd.concat([old[~old.record.isin(new_real.record)], new_real]).drop_duplicates("record", keep="last")
    else:
        merged = new
    merged.sort_values("record").to_csv(log_file, index=False)
    return new


Overwriting /content/drive/MyDrive/sleep_project/cap_preprocess.py


In [ ]:
# Load the module
sys.path.insert(0, ROOT)
import cap_preprocess as cp
importlib.reload(cp)
print("cap_preprocess", cp.CODE_VERSION, "loaded")

cap_preprocess v3 loaded


In [ ]:
# Record list, annotations and download status
records = [r.strip().replace(".edf", "") for r in requests.get(S3 + "RECORDS", timeout=60).text.split()]
for f in ["SHA256SUMS.txt", "gender-age.xlsx"] + [f"{r}.txt" for r in records]:
    if not (ANN / f).exists():
        (ANN / f).write_bytes(requests.get(S3 + f, timeout=60).content)
print(len(records), "records |", len(list(ANN.glob("*.txt"))) - 1, "annotation files")

size_file = Path(ROOT) / "server_sizes.json"
if size_file.exists():
    sizes = json.loads(size_file.read_text())
else:
    with ThreadPoolExecutor(16) as ex:
        sizes = dict(ex.map(lambda r: (r, int(requests.head(S3 + f"{r}.edf", timeout=60).headers["Content-Length"])), records))
    size_file.write_text(json.dumps(sizes))
TOTAL = sum(sizes.values())
local_size = lambda r: (EDF / f"{r}.edf").stat().st_size if (EDF / f"{r}.edf").exists() else 0
todo = [r for r in records if local_size(r) != sizes[r]]
print(f"{len(records) - len(todo)}/{len(records)} EDF files complete | to download: {len(todo)}")

108 records | 108 annotation files
108/108 EDF files complete | to download: 0


In [ ]:
# Download missing EDF files (resumable)
stop = threading.Event()

def download(r):
    path, total = EDF / f"{r}.edf", sizes[r]
    for attempt in range(20):
        if stop.is_set(): return r, "stopped"
        have = local_size(r)
        if have == total: return r, "ok"
        if have > total: path.unlink(); have = 0
        try:
            with requests.get(S3 + f"{r}.edf", headers={"Range": f"bytes={have}-"}, stream=True, timeout=(15, 120)) as resp:
                resp.raise_for_status()
                with open(path, "ab") as f:
                    for chunk in resp.iter_content(1 << 22):
                        if stop.is_set(): return r, "stopped"
                        f.write(chunk)
        except Exception:
            time.sleep(min(60, 5 * (attempt + 1)))
    return r, "ok" if local_size(r) == total else "FAILED"

def monitor():
    t0, b0 = time.time(), sum(local_size(r) for r in records)
    while not stop.wait(300):
        b = sum(local_size(r) for r in records)
        done = sum(local_size(r) == sizes[r] for r in records)
        avg = (b - b0) / (time.time() - t0) / 1e6
        eta = (TOTAL - b) / 1e6 / avg / 60 if avg > 0 else float("nan")
        print(f"[{time.strftime('%H:%M:%S')}] files {done}/{len(records)} | {b/1e9:.1f}/{TOTAL/1e9:.1f} GB | "
              f"{avg:.1f} MB/s | ETA {eta:.0f} min", flush=True)

if todo:
    threading.Thread(target=monitor, daemon=True).start()
    try:
        with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
            for fut in as_completed([ex.submit(download, r) for r in todo]):
                r, s = fut.result()
                if s != "ok": print(f"  {r}: {s}", flush=True)
    except KeyboardInterrupt:
        print("stopped - run this cell again to resume")
    finally:
        stop.set()
print(f"EDF complete: {sum(local_size(r) == sizes[r] for r in records)}/{len(records)}")

EDF complete: 108/108


In [ ]:
# SHA-256 check of the EDF files (cached)
sums = dict(l.split()[::-1] for l in (ANN / "SHA256SUMS.txt").read_text().splitlines() if l.strip())
cache_file = Path(ROOT) / "checksum_ok.json"
cache = json.loads(cache_file.read_text()) if cache_file.exists() else {}
bad = []
for i, r in enumerate(records, 1):
    p = EDF / f"{r}.edf"
    if not p.exists(): bad.append(r); continue
    if cache.get(r) == p.stat().st_size: continue
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() == sums.get(f"{r}.edf"): cache[r] = p.stat().st_size
    else: bad.append(r)
    cache_file.write_text(json.dumps(cache))
    print(f"verified {r} ({i}/{len(records)})", flush=True)
print("files OK:", len(cache), "/", len(records), "| BAD or missing:", bad or "none")

verified brux1 (1/108)
verified brux2 (2/108)
verified ins1 (3/108)
verified ins2 (4/108)
verified ins3 (5/108)
verified ins4 (6/108)
verified ins5 (7/108)
verified ins6 (8/108)
verified ins7 (9/108)
verified ins8 (10/108)
verified ins9 (11/108)
verified n10 (12/108)
verified n11 (13/108)
verified n12 (14/108)
verified n13 (15/108)
verified n14 (16/108)
verified n15 (17/108)
verified n16 (18/108)
verified n1 (19/108)
verified n2 (20/108)
verified n3 (21/108)
verified n4 (22/108)
verified n5 (23/108)
verified n6 (24/108)
verified n7 (25/108)
verified n8 (26/108)
verified n9 (27/108)
verified narco1 (28/108)
verified narco2 (29/108)
verified narco3 (30/108)
verified narco4 (31/108)
verified narco5 (32/108)
verified nfle10 (33/108)
verified nfle11 (34/108)
verified nfle12 (35/108)
verified nfle13 (36/108)
verified nfle14 (37/108)
verified nfle15 (38/108)
verified nfle16 (39/108)
verified nfle17 (40/108)
verified nfle18 (41/108)
verified nfle19 (42/108)
verified nfle1 (43/108)
verified nfl

In [ ]:
# Channel inventory: records with EEG, chin EMG and ECG
inv_file = Path(ROOT) / "cap_inventory.csv"
inv = pd.read_csv(inv_file) if inv_file.exists() else None
if inv is None or "EMG_chin" not in inv.columns:
    rows = []
    for r in records:
        h = cp.read_header(EDF / f"{r}.edf")
        idx = {k: cp.pick(h["labels"], rule) for k, rule in cp.RULES.items()}
        rows.append({"record": r, "group": cp.GROUP_NAMES[re.match(r"[a-z]+", r).group()],
                     "duration_h": h["n_rec"] * h["rec_dur"] / 3600,
                     "EEG": idx["eeg"] is not None, "EMG_chin": idx["emg"] is not None, "ECG": idx["ecg"] is not None,
                     "eeg_fs": h["spr"][idx["eeg"]] / h["rec_dur"] if idx["eeg"] is not None else np.nan,
                     "labels": "|".join(h["labels"])})
    inv = pd.DataFrame(rows)
    inv["usable"] = inv.EEG & inv.EMG_chin & inv.ECG & ~inv.record.isin(cp.EXCLUDE)
    inv.to_csv(inv_file, index=False)
print(inv.groupby("group")[["EEG", "EMG_chin", "ECG", "usable"]].sum().astype(int).to_string())
usable = sorted(inv.query("usable").record)
print(f"\nusable records: {len(usable)} / {len(inv)}")

       EEG  EMG_chin  ECG  usable
group                            
brux     2         2    2       2
ins      9         9    9       9
n       16        13   15      13
narco    5         5    5       5
nfle    40        40   40      39
plm     10        10   10      10
rbd     22        22   22      22
sdb      4         4    4       4

usable records: 104 / 108


In [ ]:
# Preprocess all usable records -> cap_processed/*.npz
if FORCE_REPROCESS:
    shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir(parents=True, exist_ok=True)
log = cp.run(usable, str(EDF), str(ANN), str(ANN / "gender-age.xlsx"), str(OUT))
print("\n", log.status.str.split(":").str[0].value_counts().to_string())

[  1/104] brux1    ok: 1772/2298 windows kept (418 of them clipped >1%)  (5s)
[  2/104] brux2    ok: 5407/5412 windows kept (740 of them clipped >1%)  (5s)
[  3/104] ins1     ok: 4709/4710 windows kept (54 of them clipped >1%)  (8s)
[  4/104] ins2     ok: 5026/5028 windows kept (27 of them clipped >1%)  (5s)
[  5/104] ins3     ok: 3743/3760 windows kept (368 of them clipped >1%)  (5s)
[  6/104] ins4     ok: 3929/3930 windows kept (1397 of them clipped >1%)  (3s)
[  7/104] ins5     ok: 4799/4812 windows kept (1313 of them clipped >1%)  (8s)
[  8/104] ins6     ok: 3372/3378 windows kept (357 of them clipped >1%)  (10s)
[  9/104] ins7     ok: 5238/5238 windows kept (847 of them clipped >1%)  (9s)
[ 10/104] ins8     ok: 3600/3612 windows kept (261 of them clipped >1%)  (8s)
[ 11/104] ins9     ok: 2331/2376 windows kept (80 of them clipped >1%)  (9s)
[ 12/104] n1       ok: 6600/6606 windows kept (1340 of them clipped >1%)  (15s)
[ 13/104] n10      ok: 4716/4716 windows kept (2160 of them cl

In [ ]:
# Summary of the processed data
rows, st_rows = [], []
for f in sorted(glob.glob(f"{OUT}/*.npz")):
    d = np.load(f); m = json.loads(str(d["meta"]))
    clip = (d["eeg_clip"].astype(float) > 0.01) | (d["emg_clip"].astype(float) > 0.01)
    rows.append({"record": m["record"], "group": m["group"], "age": m["age"], "sex": m["sex"],
                 "total": m["windows_total"], "kept": m["windows_kept"], "clipped": int(clip.sum()),
                 "hr_med": float(np.median(d["hr"])), "rmssd_med": float(np.median(d["rmssd"])), **m["reasons"]})
    for s in np.unique(d["stage"]):
        k = d["stage"] == s
        st_rows.append({"stage": str(s), "windows": int(k.sum()), "clipped": int(clip[k].sum())})
summary = pd.DataFrame(rows)
summary["clipped_%"] = (100 * summary.clipped / summary.kept).round(1)
summary.to_csv(Path(ROOT) / "processed_summary.csv", index=False)

print(f"records: {len(summary)} | windows kept: {summary.kept.sum():,} of {summary.total.sum():,} "
      f"| clipped >1%: {summary.clipped.sum():,} ({summary.clipped.sum()/summary.kept.sum():.1%})\n")
print("hard rejections (all records):"); print(summary[["flat_eeg", "flat_emg", "flat_ecg", "no_label"]].sum().to_string(), "\n")
st = pd.DataFrame(st_rows).groupby("stage").sum(); st["clipped_%"] = (100 * st.clipped / st.windows).round(1)
print("clipping by sleep stage:"); print(st.to_string(), "\n")
print("per group:"); print(summary.groupby("group").agg(records=("record", "count"), windows=("kept", "sum"),
      hr_median=("hr_med", "median"), rmssd_median=("rmssd_med", "median"), clipped_pct=("clipped_%", "median")).round(1).to_string(), "\n")
print("records with most clipping:")
print(summary.sort_values("clipped_%", ascending=False)[["record", "group", "kept", "clipped_%"]].head(15).to_string(index=False))
print("\nskipped records:")
plog = pd.read_csv(OUT / "preprocess_log.csv")
print(plog[plog.status.str.startswith("skip")].to_string(index=False))

records: 103 | windows kept: 483,170 of 505,564 | clipped >1%: 92,724 (19.2%)

hard rejections (all records):
flat_eeg       26
flat_emg       31
flat_ecg        9
no_label    22392 

clipping by sleep stage:
       windows  clipped  clipped_%
stage                             
N1       25505     1360        5.3
N2      212498    32102       15.1
N3      142694    53371       37.4
REM     102473     5891        5.7 

per group:
            records  windows  hr_median  rmssd_median  clipped_pct
group                                                             
Bruxism           2     7179       62.9          43.5         18.6
Healthy          13    69386       63.1          62.4          4.3
Insomnia          9    36747       69.4          43.3          9.8
NFLE             38   190401       61.0          49.5         29.0
Narcolepsy        5    20688       65.3          47.5         27.9
PLM              10    42489       66.5          24.9         16.6
RBD              22   101987    

In [ ]:
# Diagnosis of the excluded record nfle12 (ECG without usable R peaks)
from google.colab import drive
drive.mount("/content/drive")
import sys, importlib, numpy as np
from pathlib import Path
ROOT = "/content/drive/MyDrive/sleep_project"
EDF = Path(ROOT) / "cap_edf"
sys.path.insert(0, ROOT)
import cap_preprocess as cp
importlib.reload(cp)

r = "nfle12"
h = cp.read_header(EDF / f"{r}.edf")
i = cp.pick(h["labels"], cp.RULES["ecg"])
x, fs = cp.read_channel(EDF / f"{r}.edf", h, i)
ecg = cp.bandpass(cp.to_fs(x, fs), *cp.BANDS["ecg"])
print("channel:", h["labels"][i], "| unit:", h["unit"][i], "| fs:", fs, "| range:", h["pmin"][i], h["pmax"][i])
print("signal std (uV):", round(float(ecg.std()), 2), "| p1/p99:", np.percentile(ecg, [1, 99]).round(1))
for name, sig in [("normal", ecg), ("inverted", -ecg)]:
    rr_t, rr, ok = cp.rr_series(sig)
    hr = 60 / np.median(rr[ok]) if ok.any() else float("nan")
    print(f"{name:9s} beats: {len(rr)+1:6d} | valid RR: {ok.mean():.1%} | median HR: {hr:.1f}")
print("all channel labels:", h["labels"])
drive.flush_and_unmount()

Mounted at /content/drive
channel: ECG1-ECG2 | unit: mV | fs: 512.0 | range: -0.5 0.5
signal std (uV): 0.26 | p1/p99: [-0.4  0.4]
normal    beats:  16392 | valid RR: 21.6% | median HR: 46.6
inverted  beats:  16433 | valid RR: 21.5% | median HR: 46.0
all channel labels: ['Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'F8-T4', 'T4-T6', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'F7-T3', 'T3-T5', 'C4-A1', 'ROC-LOC', 'EMG1-EMG2', 'ECG1-ECG2', 'DX1-DX2', 'SX1-SX2', 'SAO2', 'HR', 'PLETH', 'STAT']


In [ ]:
# Save everything to Drive
drive.flush_and_unmount()
print("all changes saved to Drive")

all changes saved to Drive
